# Inverse design of a metalens

*optics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/optics/metalens_inverse_design)

A flat metasurface of sub-wavelength meta-atoms imprints a phase $\varphi(x)$ on light. Light is propagated by the angular-spectrum method (one FFT), and autograd with Adam tunes every meta-atom to push as much power as possible into a focal spot, from a flat start. The result is compared with the analytic hyperbolic lens, and designing for two wavelengths at once (a batch) shows the chromatic trade-off.

**Vector form**

$$E(x, z) = \int \hat E(k_x, 0)\, e^{i k_x x + i\sqrt{k^2 - k_x^2}\,z}\,\frac{dk_x}{2\pi},\qquad E(x, 0) = A(x)\,e^{i\varphi(x)} \qquad \max_{\varphi}\ \eta = \frac{\int_{|x|<w} |E(x, f)|^2\,dx}{\int |E(x, 0)|^2\,dx}$$

**Index (tensor) form**

$$\hat E_{lq} = F_{qx}\,A_x\,e^{i\varphi_{lx}},\qquad E_{lzx} = F^{-1}_{xq}\,e^{i k_{z,lq} z_z}\,\hat E_{lq} \qquad \varphi_{lx} = 2\pi\,\tfrac{\lambda_0}{\lambda_l}\,(\theta_x \bmod 1) \qquad \eta_l = \frac{S_{lx}\,|E_{lx}(f)|^2}{|E_{lx}(0)|^2}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x,\ k_{x,q}$ | `tensors.grid(device)` | (X,), (Q,) | 2048 grid points ($\Delta x = 0.05$ μm) and their FFT wavenumbers |
| $\theta_x$ | `atoms_to_grid(theta_atoms, x, D)` | (X,) | design variable: phase delay of each meta-atom in waves at $\lambda_0$, one value per 0.25 μm atom |
| $\varphi_{lx}$ | `metasurface_phase(theta, wavelengths, lam0)` | (L, X) | phase at wavelength $\lambda_l$; $l$ is a batch of two wavelengths |
| $k_{z,lq}$ | `sqrt(k[:, None]**2 - kx[None, :]**2)` | (L, Q) | complex: imaginary (decaying) for evanescent components $/k_x/ > k$ |
| $F_{qx}\,(\cdot)_x$ | `torch.fft.fft(E0, dim=-1)` | (L, Q) | repeated $x$ ⇒ summed, done in $O(X\log X)$ by the FFT |
| $E_{lzx}$ | `propagate(E0, kz, z)` | (L, Z, X) | field at every wavelength and distance at once, by broadcasting the transfer function |
| $S_{lx}\,\vert E_{lx}\vert ^2$ | `einsum('lx,lx->l', spot, E_f.abs()**2)` | (L,) | repeated $x$ ⇒ summed: power inside the focal spot $/x/ < \lambda_l f / D$ |
| $\partial \eta / \partial \theta_x$ | `loss.backward()` | (n_atoms,) | autograd through the FFTs: the adjoint method for free |

The analytic metalens puts every path from the lens to the focus in phase, $\varphi(x) = -k(\sqrt{x^2 + f^2} - f)$, wrapped into $[0, 2\pi)$ because a meta-atom can only delay light by up to one wave (visible metalenses of this kind were demonstrated by Khorasaninejad et al., Science 2016). Here no formula is given to the optimiser: it starts from a flat surface, computes the focal-spot power $\eta$ with an FFT-based propagator, and gets the gradient with respect to all meta-atom phases from one backward pass, the same information the adjoint method gives in photonic inverse design. For a single wavelength and a single focal point the best phase is close to the hyperbolic one, and the optimiser rediscovers it (it may do slightly better on $\eta$, because the spot has a finite width and the scalar propagator is not exactly the ray picture). The interesting case is two wavelengths: in the toy model each meta-atom adds a fixed optical path wrapped into one wave, so at $\lambda_1 \neq \lambda_0$ the phase is scaled by $\lambda_0/\lambda_1$ and the wraps no longer line up. The hyperbolic lens then focuses red light at a different distance (diffractive chromatic aberration). With design 'dual' both wavelengths share the loss and the optimiser finds a compromise: better at $\lambda_1$, worse at $\lambda_0$. Simplifications: scalar 1D light, a thin-element phase mask (each meta-atom is an ideal phase shifter with no coupling between neighbours), perfect transmission, a toy dispersion model and periodic boundaries. Real inverse design solves Maxwell's equations for the meta-atoms; differentiable models of whole metasurface cameras were trained end to end by Tseng et al. ("Neural nano-optics", Nature Communications 2021), and Li et al. review inverse-designed metasurfaces (ACS Photonics 2022). How much bandwidth a flat lens can focus at once has fundamental limits (Presutti & Monticone, Optica 2020).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/optics/metalens_inverse_design/main.py
"""Main loop: optics setup → analytic hyperbolic lens → inverse design with Adam → field maps → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    lam0, lam1, f, D = params["wavelength"], params["wavelength2"], params["focal_length"], params["diameter"]
    wavelengths = torch.tensor([lam0, lam1])  # batch of two wavelengths, li = 0, 1
    optics = solve.Optics(wavelengths, lam0, f, D, dev)

    # Analytic reference: the wrapped hyperbolic phase, designed for λ0.
    centres = tensors.atom_centres(D, dev)
    theta_hyp = torch.remainder(equation.hyperbolic_theta(centres, f, lam0), 1.0)  # (n_atoms,)
    eta_hyp = optics.efficiency(theta_hyp).detach().cpu()

    # Inverse design from a flat surface. 'single': only λ0 counts; 'dual': both wavelengths count equally.
    weights = torch.tensor([1.0, 0.0] if params["design"] == "single" else [0.5, 0.5], device=dev)
    n_iter = 20 if fast_mode() else params["n_iter"]
    theta0 = torch.zeros(tensors.n_atoms(D), device=dev)
    history, snap_iters, snaps = solve.optimise(optics, theta0, weights, n_iter, params["lr"])
    eta_opt = history[-1]

    z_scan = torch.linspace(0.3 * f, 2.0 * f, 400, device=dev)
    peaks_hyp = solve.on_axis_peak(optics, theta_hyp, z_scan).tolist()
    peaks_opt = solve.on_axis_peak(optics, snaps[-1], z_scan).tolist()
    print(f"metalens: D = {D} μm, f = {f} μm, NA = {D / 2 / (D**2 / 4 + f**2) ** 0.5:.2f}, "
          f"{tensors.n_atoms(D)} meta-atoms of {tensors.PERIOD} μm, design '{params['design']}'")  # fmt: skip
    for li, lam in enumerate(wavelengths.tolist()):
        print(
            f"λ = {lam:.3f} μm   focal-spot efficiency: hyperbolic {eta_hyp[li]:6.1%}   optimised {eta_opt[li]:6.1%}"
            f"   brightest on-axis point: hyperbolic z = {peaks_hyp[li]:5.1f}, optimised z = {peaks_opt[li]:5.1f} μm"
        )

    # |E(x, z)| maps for the animation: every snapshot, both wavelengths.
    z_map = torch.linspace(0.0, 1.6 * f, 24 if fast_mode() else 110, device=dev)
    keep = torch.nonzero(optics.x.abs() < 0.75 * D).squeeze(1)[::4]  # crop and thin the x axis for drawing
    maps = torch.stack([optics.field_map(th, z_map, keep) for th in snaps]).cpu()  # (S, L, Z, X_kept)
    map_hyp = optics.field_map(theta_hyp, z_map, keep).cpu()
    return plot.animate(
        maps, map_hyp, optics.x[keep].cpu(), z_map.cpu(), snaps.cpu(), theta_hyp.cpu(), centres.cpu(),
        history, eta_hyp, snap_iters, wavelengths, f, out_dir / "metalens_inverse_design.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/optics/metalens_inverse_design/equation.py
r"""Physics → tensor translation: scalar angular-spectrum propagation and the focusing figure of merit.

A 1D metasurface at z = 0 imprints a phase φ_l(x) on a plane wave of wavelength λ_l (l = wavelength index):

    E_{l}(x, 0) = A(x) e^{i φ_l(x)},        φ_l(x) = 2π \frac{λ_0}{λ_l} (θ(x) \bmod 1)

θ(x) ∈ [0, 1) is the design variable: the phase delay of the meta-atom at x, in waves at the design wavelength λ_0.
(Toy dispersion model: a fixed optical path, wrapped into one wave. Real meta-atoms have their own dispersion.)

Angular spectrum: each plane-wave component e^{i k_x x} picks up the phase e^{i k_z z}, k_z = \sqrt{k^2 - k_x^2}:

    \hat E_{lq}(z) = e^{i k_{z,lq} z} \hat E_{lq}(0),     \hat E_{lq} = F_{qx} E_{lx}   (F = discrete Fourier transform)

Components with |k_x| > k are evanescent: k_z is imaginary and they decay. The FFT does the sum over x.

Figure of merit: the fraction of the incident power that lands in the focal spot |x| < w at z = f,

    η_l = \sum_{|x| < w} |E_l(x, f)|^2 / \sum_x |E_l(x, 0)|^2
"""

import math

import torch


def kz(kx: torch.Tensor, wavelengths: torch.Tensor) -> torch.Tensor:
    """k_{z,lq} = sqrt(k_l² - k_{x,q}²) as a complex number (imaginary for evanescent waves): (L, Q)."""
    k = 2 * math.pi / wavelengths  # (L,)
    return torch.sqrt((k[:, None] ** 2 - kx[None, :] ** 2).to(torch.complex64))


def propagate(E0: torch.Tensor, kz_lq: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    r"""E_l(x, z) for every wavelength l and distance z at once.

    E0 (L, X) complex, kz (L, Q), z (Z,)  →  (L, Z, X).   \hat E_{lzq} = e^{i k_{z,lq} z_z} \hat E_{lq}
    """
    spectrum = torch.fft.fft(E0, dim=-1)  # (L, Q)
    H = torch.exp(1j * kz_lq[:, None, :] * z[None, :, None])  # (L, Z, Q) transfer function
    return torch.fft.ifft(spectrum[:, None, :] * H, dim=-1)


def metasurface_phase(theta: torch.Tensor, wavelengths: torch.Tensor, lam0: float) -> torch.Tensor:
    """φ_l(x) = 2π (λ0 / λ_l) (θ(x) mod 1): theta (X,) → (L, X). The wrap is what makes a metalens chromatic."""
    return 2 * math.pi * (lam0 / wavelengths)[:, None] * torch.remainder(theta, 1.0)[None, :]


def hyperbolic_theta(x: torch.Tensor, f: float, lam0: float) -> torch.Tensor:
    r"""Analytic lens: every path from x to the focus is in phase, φ(x) = -k_0 (\sqrt{x^2 + f^2} - f).  In waves."""
    return -(torch.sqrt(x**2 + f**2) - f) / lam0


def field_at_surface(aperture: torch.Tensor, phase: torch.Tensor) -> torch.Tensor:
    """E_l(x, 0) = A(x) e^{i φ_l(x)}: (L, X) complex."""
    return aperture[None, :] * torch.exp(1j * phase)


def focal_efficiency(E_f: torch.Tensor, E0: torch.Tensor, spot: torch.Tensor) -> torch.Tensor:
    """η_l = Σ_x spot_{lx} |E_l(x, f)|² / Σ_x |E_l(x, 0)|²  (repeated x ⇒ summed): (L,)."""
    return torch.einsum("lx,lx->l", spot, E_f.abs() ** 2) / torch.einsum("lx->l", E0.abs() ** 2)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/optics/metalens_inverse_design/tensors.py
"""Build the concrete tensors: the x grid, its wavenumbers, the aperture, focal-spot masks and the meta-atom map."""

import math

import torch

WIDTH = 102.4  # simulation window (μm); the field is periodic in it, so keep the light away from its edges
N_GRID = 2048  # grid points → Δx = 0.05 μm, well below λ/2
PERIOD = 0.25  # meta-atom period (μm) = 5 grid points: one design value per meta-atom


def grid(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """x (X,) centred on 0 and the matching FFT wavenumbers k_x (Q = X,), in μm and 1/μm."""
    dx = WIDTH / N_GRID
    x = (torch.arange(N_GRID, device=device) - N_GRID // 2) * dx
    kx = 2 * math.pi * torch.fft.fftfreq(N_GRID, d=dx).to(device)
    return x, kx


def aperture(x: torch.Tensor, diameter: float) -> torch.Tensor:
    """A(x) = 1 on the lens |x| < D/2, 0 elsewhere (an opaque screen around it): (X,)."""
    return (x.abs() < diameter / 2).float()


def n_atoms(diameter: float) -> int:
    return int(round(diameter / PERIOD))


def atoms_to_grid(theta_atoms: torch.Tensor, x: torch.Tensor, diameter: float) -> torch.Tensor:
    """Spread one design value per meta-atom over its grid points: (n_atoms,) → (X,) (zero outside the lens)."""
    per_atom = int(round(PERIOD / (WIDTH / N_GRID)))
    inside = theta_atoms.repeat_interleave(per_atom)  # (n_atoms · per_atom,)
    start = int(torch.searchsorted(x, torch.tensor(-diameter / 2, device=x.device)))
    out = torch.zeros_like(x, dtype=theta_atoms.dtype)
    return torch.cat([out[:start], inside, out[start + len(inside) :]])


def atom_centres(diameter: float, device: torch.device) -> torch.Tensor:
    """x position of the centre of every meta-atom (n_atoms,)."""
    n = n_atoms(diameter)
    return (torch.arange(n, device=device) + 0.5) * PERIOD - diameter / 2


def spot_masks(x: torch.Tensor, wavelengths: torch.Tensor, f: float, diameter: float) -> torch.Tensor:
    """1 inside the focal spot |x| < w_l = λ_l f / D (the first zero of a diffraction-limited focus): (L, X)."""
    w = wavelengths * f / diameter  # (L,)
    return (x[None, :].abs() < w[:, None]).float()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/optics/metalens_inverse_design/solve.py
"""Inverse design: maximise the focal-spot power with autograd + Adam, starting from a flat (do-nothing) surface."""

import torch

from problems._common.progress import steps

from . import tensors
from .equation import field_at_surface, focal_efficiency, kz, metasurface_phase, propagate


class Optics:
    """Everything fixed during the optimisation: grid, aperture, wavelengths, focal length, spot masks."""

    def __init__(self, wavelengths: torch.Tensor, lam0: float, f: float, diameter: float, device: torch.device):
        self.x, kx = tensors.grid(device)
        self.wavelengths = wavelengths.to(device)
        self.lam0, self.f, self.diameter = lam0, f, diameter
        self.aperture = tensors.aperture(self.x, diameter)
        self.kz = kz(kx, self.wavelengths)  # (L, Q)
        self.spot = tensors.spot_masks(self.x, self.wavelengths, f, diameter)  # (L, X)

    def surface_field(self, theta_atoms: torch.Tensor) -> torch.Tensor:
        theta = tensors.atoms_to_grid(theta_atoms, self.x, self.diameter)  # (X,)
        return field_at_surface(self.aperture, metasurface_phase(theta, self.wavelengths, self.lam0))  # (L, X)

    def efficiency(self, theta_atoms: torch.Tensor) -> torch.Tensor:
        """η_l for every wavelength (L,), differentiable with respect to theta_atoms."""
        E0 = self.surface_field(theta_atoms)
        z = torch.tensor([self.f], device=self.x.device)
        E_f = propagate(E0, self.kz, z)[:, 0]  # (L, X) field in the focal plane
        return focal_efficiency(E_f, E0, self.spot)

    @torch.no_grad()
    def field_map(self, theta_atoms: torch.Tensor, z: torch.Tensor, keep: torch.Tensor) -> torch.Tensor:
        """|E_l(x, z)| on the z grid and the x points selected by keep: (L, Z, X_kept)."""
        return propagate(self.surface_field(theta_atoms), self.kz, z)[..., keep].abs()


def optimise(optics: Optics, theta0: torch.Tensor, weights: torch.Tensor, n_iter: int, lr: float, n_snap: int = 60):
    """Adam on loss = -Σ_l weights_l η_l. θ is in waves, so lr = 0.02 moves a meta-atom by ~7° per step.

    Returns the efficiency history (n_iter + 1, L), the iterations of the snapshots and θ at each snapshot.
    """
    theta = theta0.clone().requires_grad_()
    opt = torch.optim.Adam([theta], lr=lr)
    # Snapshots closer together early on, where the design changes fastest.
    snap_iters = sorted({round(n_iter * (s / max(1, n_snap - 1)) ** 2) for s in range(n_snap)})
    history, snaps = [], []
    for it in steps(n_iter + 1, "Adam steps"):
        eta = optics.efficiency(theta)
        history.append(eta.detach().cpu())
        if it in snap_iters:
            snaps.append(theta.detach().clone())
        if it == n_iter:
            break
        loss = -(weights * eta).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return torch.stack(history), snap_iters, torch.stack(snaps)


@torch.no_grad()
def on_axis_peak(optics: Optics, theta_atoms: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    """Distance z of the brightest point on the axis x = 0, for every wavelength: (L,)."""
    centre = optics.x.abs().argmin()
    intensity = propagate(optics.surface_field(theta_atoms), optics.kz, z)[..., centre].abs() ** 2  # (L, Z)
    return z[intensity.argmax(dim=-1)]


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/optics/metalens_inverse_design/plot.py
"""Animate the design: |E(x, z)| at both wavelengths, the meta-atom phases and the focal efficiency vs iteration."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

GREY = "#6b7280"
COLORS = ["#16a34a", "#dc2626"]  # green and red for the two wavelengths


def animate(
    maps: torch.Tensor,
    map_hyp: torch.Tensor,
    x: torch.Tensor,
    z: torch.Tensor,
    snaps: torch.Tensor,
    theta_hyp: torch.Tensor,
    centres: torch.Tensor,
    history: torch.Tensor,
    eta_hyp: torch.Tensor,
    snap_iters: list[int],
    wavelengths: torch.Tensor,
    f: float,
    path: Path,
) -> Path:
    maps_np = maps.numpy()  # (S, L, Z, X)
    xs, zs = x.numpy(), z.numpy()
    lams = wavelengths.tolist()
    hist = history.numpy()  # (n_iter + 1, L)
    th = np.remainder(snaps.numpy(), 1.0)  # (S, n_atoms)
    xc = centres.numpy()
    vmax = [max(maps_np[:, li].max(), map_hyp[li].max().item()) for li in range(2)]

    fig = plt.figure(figsize=(8.4, 5.4))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.25, 1])
    axes_map = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])]
    ax_p = fig.add_subplot(grid[1, 0])
    ax_e = fig.add_subplot(grid[1, 1])
    cmap = plt.get_cmap("magma", 14)

    images = []
    for li, ax in enumerate(axes_map):
        im = ax.imshow(maps_np[0, li], origin="lower", aspect="auto", cmap=cmap, vmin=0, vmax=vmax[li],
                       extent=[xs[0], xs[-1], zs[0], zs[-1]], interpolation="nearest")  # fmt: skip
        ax.plot([0], [f], "+", color="white", ms=10, mew=1.5)
        ax.set_title(f"|E(x, z)| at λ = {lams[li]:.3f} μm", fontsize=10, color=COLORS[li])
        ax.set_xlabel("x (μm)", fontsize=9)
        ax.grid(False)
        ax.tick_params(labelsize=8)
        images.append(im)
    axes_map[0].set_ylabel("z (μm)", fontsize=9)

    # meta-atom phases θ (in waves at λ0)
    ax_p.plot(xc, theta_hyp.numpy(), ".", color=GREY, ms=2.5, label="hyperbolic (wrapped)")
    (dots,) = ax_p.plot(xc, th[0], ".", color="#2563eb", ms=3.5, label="optimised")
    ax_p.set_ylim(-0.05, 1.3)
    ax_p.set_xlabel("x (μm)", fontsize=9)
    ax_p.set_ylabel("θ = φ / 2π at λ0", fontsize=9)
    ax_p.set_title("meta-atom phase delays", fontsize=10)
    ax_p.legend(loc="upper center", fontsize=7, ncol=2)
    ax_p.tick_params(labelsize=8)

    # focal-spot efficiency vs iteration
    it = np.arange(len(hist))
    ax_e.set_xlim(0, max(1, len(hist) - 1))
    ax_e.set_ylim(0, max(hist.max(), eta_hyp.max().item()) * 1.2)
    lines = []
    for li in range(2):
        ax_e.axhline(eta_hyp[li].item(), color=COLORS[li], ls="--", lw=1, alpha=0.8)
        lines.append(ax_e.plot([], [], color=COLORS[li], lw=1.8, label=f"{lams[li]:.3f} μm")[0])
    ax_e.plot([], [], ls="--", color=GREY, lw=1, label="hyperbolic")
    ax_e.set_xlabel("Adam iteration", fontsize=9)
    ax_e.set_title("power in focal spot η", fontsize=10)
    ax_e.legend(loc="lower right", fontsize=7)
    ax_e.tick_params(labelsize=8)
    label = ax_e.text(0.03, 0.94, "", transform=ax_e.transAxes, va="top", fontsize=8)

    def update(s: int):
        for li in range(2):
            images[li].set_data(maps_np[s, li])
            lines[li].set_data(it[: snap_iters[s] + 1], hist[: snap_iters[s] + 1, li])
        dots.set_ydata(th[s])
        eta = hist[snap_iters[s]]
        label.set_text(f"iteration {snap_iters[s]}:  η = {eta[0]:.0%} / {eta[1]:.0%}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(snaps), 60), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'design': 'dual',
    'wavelength': 0.532,
    'wavelength2': 0.633,
    'focal_length': 30.0,
    'diameter': 40.0,
    'n_iter': 300,
    'lr': 0.02
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.optics.metalens_inverse_design")]:
    del sys.modules[name]
main = importlib.import_module("problems.optics.metalens_inverse_design.main")
problem_dir = Path("problems/optics/metalens_inverse_design")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
